In [1]:
"""
fix_quantity_g.py
==================
Tìm tất cả dish_ingredient có quantity_g > 1500 (khả năng nhập sai)
→ Build prompt (dish_title + ingredient + quantity_g gốc) → chia 100 items/file
→ AI ước tính lại quantity_g đúng → import update bảng dish_ingredient
"""

import sqlite3
import json
import math
from pathlib import Path

DB_PATH    = "./cookpad_clean.db"   # <-- sửa đường dẫn DB
PROMPT_DIR = "./prompts_fix_quantity"
FILLED_DIR = "./filled_fix_quantity"
BATCH_SIZE = 30

PROMPT_TEMPLATE = """\
# ROLE
Bạn là chuyên gia ẩm thực Việt Nam với kinh nghiệm định lượng nguyên liệu chính xác.

# BỐI CẢNH
Dữ liệu khối lượng (quantity_g) dưới đây bị nhập sai do lỗi hệ thống (ví dụ: dư số 0, nhầm đơn vị).
Hãy ước tính lại khối lượng thực tế cho **1 công thức nấu ăn gia đình (2–4 người ăn)**.

# QUY TẮC ĐỊNH LƯỢNG (GRAM)
- Nguyên liệu chính (Thịt, Cá, Hải sản, Rau chính): 200g – 800g.
- Rau ăn kèm, củ quả phụ: 50g – 200g.
- Gia vị củ/quả (Hành, tỏi, ớt, gừng): 5g – 40g.
- Gia vị lá (Ngò, lá lốt, thì là): 5g – 20g.
- Nước dùng/Nước lọc: 500g – 2000g (tùy món canh hay kho).
- Nếu món ăn là 'Pate' hoặc 'Chà bông' (đồ dự trữ), khối lượng nguyên liệu chính có thể lên tới 1000g - 1500g.

# LƯU Ý QUAN TRỌNG
1. Giữ tính logic: Nếu món là "Cá kho", cá phải nặng hơn hành/tỏi.
2. Trả về số nguyên (int).
3. KHÔNG trả về kết quả > 2000g trừ trường hợp đặc biệt là nước dùng.

# OUTPUT FORMAT
Chỉ trả về JSON array, không kèm văn bản thừa:
[
  {{
    "id": <int>,
    "dish_title": "<string>",
    "ingredient_name": "<string>",
    "fixed_quantity_g": <int>
  }}
]

## BATCH {batch_num}/{total_batches}
{payload_json}
"""

# ══════════════════════════════════════════════════════════════
# BƯỚC 1 — BUILD PROMPTS
# ══════════════════════════════════════════════════════════════

con = sqlite3.connect(DB_PATH)

rows = con.execute("""
    SELECT
        di.id,
        d.title        AS dish_title,
        i.name         AS ingredient_name,
        di.quantity_g  AS original_quantity_g
    FROM dish_ingredient di
    JOIN dishes      d ON d.id = di.recipe_id
    JOIN ingredients i ON i.id = di.ingredient_id
    WHERE di.quantity_g > 1500
    ORDER BY di.quantity_g DESC
""").fetchall()

con.close()

print(f"Records có quantity_g > 1500: {len(rows)}")

payload_all = [
    {
        "id":                  r[0],
        "dish_title":          r[1],
        "ingredient_name":     r[2],
        "original_quantity_g": r[3],
    }
    for r in rows
]

total_items   = len(payload_all)
total_batches = math.ceil(total_items / BATCH_SIZE)

print(f"Batch size             : {BATCH_SIZE}")
print(f"Số file prompt sẽ tạo : {total_batches}")

Path(PROMPT_DIR).mkdir(exist_ok=True)
Path(FILLED_DIR).mkdir(exist_ok=True)

for i in range(total_batches):
    start = i * BATCH_SIZE
    end   = min(start + BATCH_SIZE, total_items)
    batch = payload_all[start:end]

    prompt = PROMPT_TEMPLATE.format(
        batch_num     = i + 1,
        total_batches = total_batches,
        start         = start + 1,
        end           = end,
        total_items   = total_items,
        payload_json  = json.dumps(batch, ensure_ascii=False, indent=2),
    )

    out_path = Path(PROMPT_DIR) / f"prompt_fix_quantity_batch_{i+1:02d}.txt"
    out_path.write_text(prompt, encoding="utf-8")
    print(f"  ✅ {out_path.name}  ({len(batch)} records)")

print(f"\n📁 Prompts saved : {Path(PROMPT_DIR).resolve()}")
print()
print("Next steps:")
print(f"  1. Mở từng file trong '{PROMPT_DIR}/'")
print(f"  2. Paste vào Claude / GPT-4o")
print(f"  3. Lưu response thành filled_fix_quantity_batch_01.json, _02.json ... vào '{FILLED_DIR}/'")
print(f"  4. Chạy BƯỚC 2 bên dưới")


# ══════════════════════════════════════════════════════════════
# BƯỚC 2 — IMPORT JSON → UPDATE dish_ingredient.quantity_g
# Chạy sau khi có đủ file filled_fix_quantity_batch_XX.json
# ══════════════════════════════════════════════════════════════

filled_files = sorted(Path(FILLED_DIR).glob("filled_fix_quantity_batch_*.json"))

if not filled_files:
    print(f"❌ Không tìm thấy file nào trong '{FILLED_DIR}/'")
else:
    print(f"Tìm thấy {len(filled_files)} file:")
    all_data = []
    for fpath in filled_files:
        with open(fpath, encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            data = data.get("results") or list(data.values())[0]
        all_data.extend(data)
        print(f"  ✅ {fpath.name}: {len(data)} items")

    print(f"\nTổng items: {len(all_data)}")

    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    updated = 0
    skipped = 0

    for item in all_data:
        record_id      = item.get("id")
        fixed_quantity = item.get("fixed_quantity_g")

        if not record_id or fixed_quantity is None:
            skipped += 1
            continue

        try:
            fixed_quantity = int(fixed_quantity)
            if fixed_quantity <= 0:
                skipped += 1
                continue
        except (ValueError, TypeError):
            skipped += 1
            continue

        cur.execute("""
            UPDATE dish_ingredient
            SET quantity_g = ?
            WHERE id = ?
        """, (fixed_quantity, record_id))
        updated += cur.rowcount

    con.commit()
    con.close()

    print(f"\n✅ Updated : {updated}")
    print(f"⚠️  Skipped : {skipped}")

    # Verify
    con = sqlite3.connect(DB_PATH)
    row = con.execute("""
        SELECT
            COUNT(*) as still_high,
            MAX(quantity_g) as max_g,
            AVG(quantity_g) as avg_g
        FROM dish_ingredient
        WHERE quantity_g > 1500
    """).fetchone()
    con.close()

    print(f"\n=== Verify sau khi fix ===")
    print(f"  Còn quantity_g > 1500 : {row[0]} records")
    print(f"  MAX quantity_g        : {row[1]}")
    print(f"  AVG quantity_g        : {row[2]:.1f}")

Records có quantity_g > 1500: 346
Batch size             : 30
Số file prompt sẽ tạo : 12
  ✅ prompt_fix_quantity_batch_01.txt  (30 records)
  ✅ prompt_fix_quantity_batch_02.txt  (30 records)
  ✅ prompt_fix_quantity_batch_03.txt  (30 records)
  ✅ prompt_fix_quantity_batch_04.txt  (30 records)
  ✅ prompt_fix_quantity_batch_05.txt  (30 records)
  ✅ prompt_fix_quantity_batch_06.txt  (30 records)
  ✅ prompt_fix_quantity_batch_07.txt  (30 records)
  ✅ prompt_fix_quantity_batch_08.txt  (30 records)
  ✅ prompt_fix_quantity_batch_09.txt  (30 records)
  ✅ prompt_fix_quantity_batch_10.txt  (30 records)
  ✅ prompt_fix_quantity_batch_11.txt  (30 records)
  ✅ prompt_fix_quantity_batch_12.txt  (16 records)

📁 Prompts saved : D:\dream_project\daily_mate_implement\db\prompts_fix_quantity

Next steps:
  1. Mở từng file trong './prompts_fix_quantity/'
  2. Paste vào Claude / GPT-4o
  3. Lưu response thành filled_fix_quantity_batch_01.json, _02.json ... vào './filled_fix_quantity/'
  4. Chạy BƯỚC 2 bên dư